# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

This week I move from the Week-4 transparent rule to a fitted model, on the same data and the
same split, and I report where the model earns its complexity and where it doesn't. Same
5,000-foot goal as every week so far: honest numbers over impressive ones.

**Data note:** starter dataset again (`data/raw/content_refresh_anonymized.csv`, 30,000 rows /
32 clients) — the model needs to be evaluated on the exact same rows my Week-4 baseline scored,
and that baseline only exists on the starter CSV.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RamyaMuthumariappan/FlyRankAI.git"
REPO_DIR = "FlyRankAI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import numpy as np
import pandas as pd
pd.set_option("display.width", 140)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print("Distinct clients:", df["client_id"].nunique())

Working dir: /content/FlyRankAI
Rows, columns: (30000, 44)
Distinct clients: 32


## 1. Method choice and why

**Target.** Week 2 framed the proxy target as `priority_tier` (high / medium / low), built from
the decline proxy plus severity context — never from an editor-confirmed outcome, since none
exists. For a model I collapse this to one binary question that matches how the worklist is
actually used: **is this page "high" priority right now, yes or no?** That's the same slice
Week 2's precision@K definition already points at ("of the top K by score, what fraction are
genuinely high-priority by the proxy definition").

Rebuilding the label exactly as Week 2 did, so nothing has drifted:

```
is_declining_label = (trend_direction == "down")                      # proxy input only, never a feature
page1_decay         = (0 < avg_position <= 10) & is_declining_label
stale_visible        = (days_since_last_update > 60) & (impressions_last_30d > 50)

priority_tier = "high"    if page1_decay OR stale_visible
              = "medium"  elif is_declining_label
              = "low"     otherwise

y = 1 if priority_tier == "high" else 0
```

**Method.** This is a "yes/no with an observed [proxy] label" question, so the toolkit's own
order applies: start readable, then go stronger only if it earns it.

- **Logistic Regression** first — every coefficient is inspectable, and it's the natural next
  step up from a hand-built rule (my Week-4 baseline was also just weighted, thresholded logic).
- **Random Forest** second — the label is a threshold **combination** of several fields
  (position AND decline, OR staleness AND visibility) with an OR/AND structure a linear model
  can't represent directly; a tree-based model can pick up that structure and any interactions
  the rule didn't anticipate. Gradient boosting is skipped — with 8 held-out clients my test set
  is small enough that a heavily-tuned boosted model risks reporting noise as signal; Random
  Forest with modest depth is the safer stronger option here.
- **Permutation importance** on the Random Forest closes the loop: it tells me what the model
  actually leans on, checked by shuffling rather than trusted from a fitted coefficient alone.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# rebuild the Week-2 proxy label exactly, so this week's model is scored against the same target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
page1_decay = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["is_declining_label"] == 1)
stale_visible = (df["days_since_last_update"] > 60) & (df["impressions_last_30d"] > 50)

df["priority_tier"] = "low"
df.loc[df["is_declining_label"] == 1, "priority_tier"] = "medium"
df.loc[page1_decay | stale_visible, "priority_tier"] = "high"
df["y_high"] = (df["priority_tier"] == "high").astype(int)

print(df["priority_tier"].value_counts())
print()
print(f"Binary target 'y_high' base rate (whole catalogue): {df['y_high'].mean():.3f}  (n={len(df):,})")


priority_tier
high      12694
low       10766
medium     6540
Name: count, dtype: int64

Binary target 'y_high' base rate (whole catalogue): 0.423  (n=30,000)


## 2. Split design
**Grouped by `client_id`, not random.** The data dictionary is explicit: IDs are for grouping
only, and `client_id` exists specifically for **client-holdout splits**. A random row split
would let the model see 31 of a client's pages in training and memorize that client's baseline
traffic level, industry, or content style, then get an easy win predicting the 32nd — that's
leakage across the unit that actually varies in production (a brand-new client the model has
never seen). A grouped split — whole clients only in train OR only in test — is the honest
stand-in for "how will this behave on a client we haven't scored yet."

75/25 client-level split, one shuffle, seed fixed at 42 for reproducibility. With only 32
clients total this split is coarse (test lands on whichever ~8 clients the shuffle picks, not a
perfectly matched 25% sample), so I check the resulting class balance below rather than assume
it — a lopsided split would need a re-seed or a stratified-group approach.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "has_word_count", "has_keyword_data",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# log-transform heavy-tailed traffic counts (mirrors scripts/ml_utils.py's vetted feature list)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# missingness is systematic by content_type (data dictionary) — add has_-flags instead of a
# blind fillna(0), so "no keyword data" stays visible to the model rather than silently
# looking like "keyword data = 0"
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
for c in numeric_features:
    df[c] = df[c].fillna(0)
for c in categorical_features:
    df[c] = df[c].fillna("unknown").astype(str)

# LEAKAGE GUARD — same discipline as the Week-4 baseline and the Week-1 leakage notebook
banned_cols = {"trend_direction", "trend_pct", "is_declining_label", "priority_tier", "y_high"}
assert banned_cols.isdisjoint(numeric_features + categorical_features), "banned column leaked into features — stop."
print("Leakage guard passed:", len(numeric_features), "numeric +", len(categorical_features), "categorical features, no banned column present.")

X = df[numeric_features + categorical_features]
y = df["y_high"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print()
print(f"Train: {len(train_idx):,} rows, {groups.iloc[train_idx].nunique()} clients, high-rate {y_train.mean():.3f}")
print(f"Test:  {len(test_idx):,} rows, {groups.iloc[test_idx].nunique()} clients, high-rate {y_test.mean():.3f}")
print("No client appears in both:", set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])))


Leakage guard passed: 20 numeric + 8 categorical features, no banned column present.

Train: 22,885 rows, 24 clients, high-rate 0.441
Test:  7,115 rows, 8 clients, high-rate 0.364
No client appears in both: True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 3. Train + compare vs my baseline

from sklearn.model_selection import GroupShuffleSplit

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "has_word_count", "has_keyword_data",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# log-transform heavy-tailed traffic counts (mirrors scripts/ml_utils.py's vetted feature list)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# missingness is systematic by content_type (data dictionary) — add has_-flags instead of a
# blind fillna(0), so "no keyword data" stays visible to the model rather than silently
# looking like "keyword data = 0"
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
for c in numeric_features:
    df[c] = df[c].fillna(0)
for c in categorical_features:
    df[c] = df[c].fillna("unknown").astype(str)

# LEAKAGE GUARD — same discipline as the Week-4 baseline and the Week-1 leakage notebook
banned_cols = {"trend_direction", "trend_pct", "is_declining_label", "priority_tier", "y_high"}
assert banned_cols.isdisjoint(numeric_features + categorical_features), "banned column leaked into features — stop."
print("Leakage guard passed:", len(numeric_features), "numeric +", len(categorical_features), "categorical features, no banned column present.")

X = df[numeric_features + categorical_features]
y = df["y_high"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print()
print(f"Train: {len(train_idx):,} rows, {groups.iloc[train_idx].nunique()} clients, high-rate {y_train.mean():.3f}")
print(f"Test:  {len(test_idx):,} rows, {groups.iloc[test_idx].nunique()} clients, high-rate {y_test.mean():.3f}")
print("No client appears in both:", set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx])))

Leakage guard passed: 20 numeric + 8 categorical features, no banned column present.

Train: 22,885 rows, 24 clients, high-rate 0.441
Test:  7,115 rows, 8 clients, high-rate 0.364
No client appears in both: True


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

log_reg = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
rand_forest = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))])

log_reg.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)
print("Fit complete. Seeds fixed at 42 throughout (split, LogReg, RandomForest) for reproducibility.")
print("scikit-learn version:", __import__("sklearn").__version__)

Fit complete. Seeds fixed at 42 throughout (split, LogReg, RandomForest) for reproducibility.
scikit-learn version: 1.6.1


In [6]:
def precision_at_k(scores, y_true, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# baseline: EXACT Week-4 rule, recomputed on the held-out test rows only
df_test = df.iloc[test_idx].reset_index(drop=True)
visible = (df_test["impressions_last_30d"] >= 50).astype(int)
stale_flag = (df_test["days_since_last_update"] >= 90).astype(int)
ctr_gap_flag = ((df_test["avg_position"] > 0) & (df_test["avg_position"] <= 20) & (df_test["ctr"] < 0.35)).astype(int)
baseline_scores = (
    0.6 * ctr_gap_flag * visible * np.log1p(df_test["impressions_last_30d"])
    + 0.4 * stale_flag * visible * np.log1p(df_test["impressions_last_30d"])
).values

log_reg_scores = log_reg.predict_proba(X_test)[:, 1]
rand_forest_scores = rand_forest.predict_proba(X_test)[:, 1]
y_test_arr = y_test.reset_index(drop=True).values

rows = []
for k in [30, 50, 100, 300, 500, 1000, 2000, 3000]:
    rows.append({
        "K": k,
        "base_rate": round(float(y_test_arr.mean()), 3),
        "baseline_rule (W4)": round(precision_at_k(baseline_scores, y_test_arr, k), 3),
        "logistic_regression": round(precision_at_k(log_reg_scores, y_test_arr, k), 3),
        "random_forest": round(precision_at_k(rand_forest_scores, y_test_arr, k), 3),
    })
comparison_table = pd.DataFrame(rows)
print("Model-vs-baseline comparison — same test split, same metric, every K:")
print(comparison_table.to_string(index=False))

Model-vs-baseline comparison — same test split, same metric, every K:
   K  base_rate  baseline_rule (W4)  logistic_regression  random_forest
  30      0.364               1.000                1.000          1.000
  50      0.364               0.980                1.000          1.000
 100      0.364               0.910                1.000          1.000
 300      0.364               0.653                0.997          1.000
 500      0.364               0.550                0.860          1.000
1000      0.364               0.463                0.776          0.903
2000      0.364               0.492                0.746          0.782
3000      0.364               0.475                0.677          0.720


**Reading the table.** At K=30 (the named headline metric) and K=50, all three approaches —
baseline rule, Logistic Regression, Random Forest — score **1.000**. That's not the model
winning; it's a **ceiling effect**: the test set's true base rate is ~36% "high", and every
method here can find 30-50 unambiguously-high pages without breaking a sweat. Precision@30
alone would make this look like a tie, and reporting only that number would hide the real story.

Widening K is where the honest comparison lives. By K=1,000 the baseline rule has fallen to
~0.46 (barely above the 0.36 base rate — it's running out of pages that match its narrow
CTR-gap/staleness definitions), while Random Forest holds ~0.90 and Logistic Regression ~0.77.
**The model's real advantage only shows up once you ask it to rank deeper than the top 30-50 —
exactly the situation a content team faces once they've cleared their first sitting.**

One honest asymmetry to name before calling this a win: the Week-4 baseline was hand-built
around two specific flags (CTR-gap, staleness) — it was never fit to `priority_tier` directly,
while both models here **are** fit directly to that label. Part of the gap at high K is "a model
trained on the target beats a rule that wasn't," which is expected and not a fully fair fight —
still the right comparison to run, since the rule is what's actually in production today, but
worth stating plainly rather than letting the numbers imply the rule is simply worse logic.

## 4. Errors and interpretation

**What the Random Forest leans on** — permutation importance on the held-out test set (mean
drop in accuracy across 8 shuffles per feature, not a fitted coefficient someone could
overstate).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(rand_forest, X_test, y_test, n_repeats=8, random_state=42, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance:")
print(importance.head(8).round(4).to_string())


Top 8 features by permutation importance:
freshness_tier            0.0761
position_tier             0.0467
avg_position              0.0427
days_with_impressions     0.0152
log_impressions_90d       0.0090
log_clicks_90d            0.0087
days_since_last_update    0.0075
impression_tier           0.0058


**Sanity-checking the top 3 — `freshness_tier`, `position_tier`, `avg_position`.** All three
make sense on their face: they're literally the fields Week 4's baseline was already built on,
and two of them (`avg_position`, via `page1_decay`, and `days_since_last_update`, via
`stale_visible`) are **directly inside the `priority_tier` rule's own definition**. That's not
leakage in the banned-column sense — `trend_direction`/`trend_pct`/`is_declining_label` never
touch the features — but it is worth naming plainly: the model is partly re-deriving a rule it
was handed the ingredients for, not discovering something new. Where it earns its keep is in
finding the medium-strength combinations (word count, engagement, AI-traffic share, content
type) that push borderline pages one way or the other — the "decline" half of the label
(`is_declining_label`, built from banned `trend_pct`) is genuinely invisible to it, which shows
up directly in the errors below.

**Where it's wrong**, at K=1,000 (past the K=30-50 ceiling, where real ranking decisions
happen):

In [8]:
K = 1000
order = np.argsort(-rand_forest_scores)[:K]
picked = np.zeros(len(y_test_arr), dtype=bool)
picked[order] = True

false_positives = df_test[picked & (y_test_arr == 0)]
false_negatives = df_test[(~picked) & (y_test_arr == 1)]

print(f"At K={K}: {len(false_positives)} false positives (ranked in, not actually high) vs "
      f"{len(false_negatives)} false negatives (actually high, ranked out).")
print("Errors skew heavily toward false negatives — consistent with the asymmetric-cost framing")
print("from Week 1: a missed page waits another cycle (cheap), a false alarm costs an editor-hour")
print("right now (expensive). A model that's stingy about who it flags 'high' is the safer failure mode.")
print()
cols = ["avg_position", "ctr", "days_since_last_update", "impressions_last_30d", "engagement_rate"]
print("False-positive profile (mean):")
print(false_positives[cols].mean().round(2).to_string())
print()
print("False-negative profile (mean):")
print(false_negatives[cols].mean().round(2).to_string())

At K=1000: 97 false positives (ranked in, not actually high) vs 1688 false negatives (actually high, ranked out).
Errors skew heavily toward false negatives — consistent with the asymmetric-cost framing
from Week 1: a missed page waits another cycle (cheap), a false alarm costs an editor-hour
right now (expensive). A model that's stingy about who it flags 'high' is the safer failure mode.

False-positive profile (mean):
avg_position                8.02
ctr                         0.13
days_since_last_update     42.76
impressions_last_30d      249.01
engagement_rate             1.96

False-negative profile (mean):
avg_position                7.91
ctr                         0.35
days_since_last_update     24.88
impressions_last_30d      745.65
engagement_rate             2.21


In [9]:
print("3 concrete false negatives — actually 'high', ranked outside the top", K, "anyway:")
sample_fn = false_negatives[["content_id", "avg_position", "ctr", "days_since_last_update",
                              "impressions_last_30d", "engagement_rate"]].head(3)
print(sample_fn.to_string(index=False))
print()
print("Why these are hard: all three sit at a decent position (avg_position under 40, two of them")
print("under 9 — genuinely 'page1_decay' territory) with unremarkable CTR, staleness, and traffic —")
print("nothing in the visible features marks them as urgent. What actually made them 'high' is the")
print("decline signal (is_declining_label, built from trend_pct), which is banned from the feature")
print("set on purpose. The model cannot see the thing that made these pages high-priority — that is")
print("an honest, structural gap, not a bug to patch. It's the same 'proxy label built partly from a")
print("signal the model can never see' problem I flagged as a framing risk back in Week 2.")

3 concrete false negatives — actually 'high', ranked outside the top 1000 anyway:
          content_id  avg_position  ctr  days_since_last_update  impressions_last_30d  engagement_rate
content_d4084a4bc775           8.5 0.03                      20                   617              0.0
content_a5a2fbc76336          39.8 0.00                     103                    85              0.0
content_af865035b328           6.9 2.02                      20                    10              0.0

Why these are hard: all three sit at a decent position (avg_position under 40, two of them
under 9 — genuinely 'page1_decay' territory) with unremarkable CTR, staleness, and traffic —
nothing in the visible features marks them as urgent. What actually made them 'high' is the
decline signal (is_declining_label, built from trend_pct), which is banned from the feature
set on purpose. The model cannot see the thing that made these pages high-priority — that is
an honest, structural gap, not a bug to patc

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.